In [1]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

conn = sqlite3.connect("../retailiq.db")
df = pd.read_sql("SELECT * FROM customer_features", conn)
conn.close()

feature_cols = [
    "recency_days", "frequency", "repeat_order_count",
    "avg_days_between_orders", "avg_review_score", "review_count",
    "avg_delivery_days", "avg_delay_days"
]
target = "monetary"

model_df = df[feature_cols + [target]].dropna()
X = model_df[feature_cols]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [2]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8, 10],
    "min_samples_leaf": [1, 5, 10],
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(
    rf, param_grid, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV MAE:", round(-grid_search.best_score_, 2))

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best params: {'max_depth': 8, 'min_samples_leaf': 10, 'n_estimators': 100}
Best CV MAE: 110.87


In [3]:
best_model = grid_search.best_estimator_
preds = best_model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"Tuned model — MAE: {round(mae, 2)}, R²: {round(r2, 3)}")

# Compare against your Day 9 baseline numbers (write these down from yesterday)
baseline_mae = None  # <-- fill in your actual Day 9 MAE here
baseline_r2 = None   # <-- fill in your actual Day 9 R² here
if baseline_mae:
    print(f"Baseline model — MAE: {baseline_mae}, R²: {baseline_r2}")
    print(f"Improvement: {round(100*(baseline_mae-mae)/baseline_mae, 1)}% MAE reduction")

Tuned model — MAE: 110.65, R²: 0.035


In [4]:
import joblib
joblib.dump(best_model, "../models_clv.pkl")

['../models_clv.pkl']